In [ ]:
import os
import sys
import json
import pickle
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.config import *
from src.data_loader import build_separated_dataset

np.random.seed(SEED)
print("Baseline environment ready.")

In [ ]:
# Load dictionaries
with open(SPLIT_PATH, "r") as f: split = json.loads(f.read())
with open(RAW_DIR / "metadata.json", "r") as f:
    id2bg = {m["material_id"]: float(m["band_gap"]) for m in json.loads(f.read()).get("materials", [])}

with open(DATA_DIR / "magpie_all_scaled.pkl", "rb") as f: magpie_scaled = pickle.load(f)
with open(DATA_DIR / "struct_scaled.pkl", "rb") as f: struct_scaled = pickle.load(f)
with open(DATA_DIR / "geo_scaled.pkl", "rb") as f: geo_scaled = pickle.load(f)

print("Building numpy arrays for Scikit-Learn...")
X_tr_m1, X_tr_m2, X_tr_oth, y_train = build_separated_dataset(split["train"], magpie_scaled, struct_scaled, geo_scaled, id2bg)
X_te_m1, X_te_m2, X_te_oth, y_test  = build_separated_dataset(split["test"], magpie_scaled, struct_scaled, geo_scaled, id2bg)

# Log transform the targets for training
y_tr_log = np.log1p(y_train)
y_te_log = np.log1p(y_test)

print(f"Arrays built! Training samples: {len(y_train)}")

In [ ]:
rf_estimator = RandomForestRegressor(n_estimators=50, n_jobs=None, random_state=SEED)

def apply_smart_reduction(reducer_m1, reducer_m2, name):
    print(f"-> Generating Dataset: [{name}]...")
    if isinstance(reducer_m1, PCA):
        tr_m1_red = reducer_m1.fit_transform(X_tr_m1)
        tr_m2_red = reducer_m2.fit_transform(X_tr_m2)
    else:
        tr_m1_red = reducer_m1.fit_transform(X_tr_m1, y_tr_log)
        tr_m2_red = reducer_m2.fit_transform(X_tr_m2, y_tr_log)
        
    te_m1_red = reducer_m1.transform(X_te_m1)
    te_m2_red = reducer_m2.transform(X_te_m2)
    
    return np.concatenate([tr_m1_red, tr_m2_red, X_tr_oth], axis=1), np.concatenate([te_m1_red, te_m2_red, X_te_oth], axis=1)

datasets = {
    "None (All Features)": (np.concatenate([X_tr_m1, X_tr_m2, X_tr_oth], axis=1), np.concatenate([X_te_m1, X_te_m2, X_te_oth], axis=1)),
    "Magpie RFE (24/30)": apply_smart_reduction(
    RFE(estimator=rf_estimator, n_features_to_select=24, step=5), 
    RFE(estimator=rf_estimator, n_features_to_select=30, step=5), 
    "RFE"
),
    "Magpie PCA (24/30)": apply_smart_reduction(PCA(24, random_state=SEED), PCA(30, random_state=SEED), "PCA"),
    "Magpie RF Importance (24/30)": apply_smart_reduction(SelectFromModel(rf_estimator, max_features=24), SelectFromModel(rf_estimator, max_features=30), "RF Importance")
}
print("Datasets generated.")

In [ ]:
models = {
    "Ridge Regressor": Ridge(alpha=1.0),
    "Lasso Regressor": Lasso(alpha=0.1),
    "HistGradBoost": HistGradientBoostingRegressor(random_state=SEED),
    "MLP (128, 64)": MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, random_state=SEED)
}

results = []
print("\nStarting ML Baseline Loop...")

for ds_name, (X_tr, X_te) in datasets.items():
    for mod_name, model in models.items():
        print(f"Training: [{ds_name}] + [{mod_name}]")
        
        pipe = Pipeline([("scaler", StandardScaler()), ("model", model)])
        pipe.fit(X_tr, y_tr_log)
        
        preds_log = pipe.predict(X_te)
        preds_log = np.clip(preds_log, Y_LOG_MIN, Y_LOG_MAX)
        preds = np.expm1(preds_log)
        
        results.append({
            "Reduction Method": ds_name, "Model": mod_name,
            "MAE": round(mean_absolute_error(y_test, preds), 4),
            "RMSE": round(mean_squared_error(y_test, preds)**0.5, 4),
            "R2": round(r2_score(y_test, preds), 4)
        })

df_ml = pd.DataFrame(results)
df_ml.to_csv(RESULTS_DIR / "ml_baselines_results.csv", index=False)
print(f"Saved results to {RESULTS_DIR / 'ml_baselines_results.csv'}")
display(df_ml.sort_values(by="MAE").head(10))